# Module 6 Homework

In this homework we'll put what we learned about Spark in practice.

For this homework we will be using the Yellow 2025-11 data from the official website:


In [1]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet -P data/

--2026-03-09 03:29:36--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 52.85.39.65, 52.85.39.97, 52.85.39.117, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|52.85.39.65|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘data/yellow_tripdata_2025-11.parquet.4’

yellow_tripdata_202 100%[===================>]  67.84M   174MB/s    in 0.4s    

2026-03-09 03:29:37 (174 MB/s) - ‘data/yellow_tripdata_2025-11.parquet.4’ saved [71134255/71134255]



## Question 1: Install Spark and PySpark

- Install Spark
- Run PySpark
- Create a local spark session
- Execute spark.version.

What's the output?

In [2]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder \
    .appName("LocalTest") \
    .master("local[*]") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/09 03:29:41 WARN Utils: Your hostname, codespaces-2b816e, resolves to a loopback address: 127.0.0.1; using 10.0.10.155 instead (on interface eth0)
26/03/09 03:29:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 03:29:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
# Print the Spark version
print(f"Spark Version: {spark.version}")

Spark Version: 4.1.1


## Question 2: Yellow November 2025

Read the November 2025 Yellow into a Spark Dataframe.

Repartition the Dataframe to 4 partitions and save it to parquet.

What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.

- 6MB
- 25MB
- 75MB
- **100MB**

In [9]:
from pyspark.sql import SparkSession
import os

# Initialize Spark
spark = SparkSession.builder \
    .appName("YellowTaxiNov2025") \
    .master("local[*]") \
    .config("spark.driver.extraJavaOptions", 
            "-Djava.security.manager=allow " +
            "--add-opens=java.base/java.util=ALL-UNNAMED " +
            "--add-opens=java.base/java.lang=ALL-UNNAMED " +
            "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED") \
    .getOrCreate()

# Path to your downloaded file
input_path = "data/raw/yellow_tripdata_2025-11.parquet"

# 1. Read the Parquet file
df = spark.read.parquet(input_path)

# 2. Repartition to 4
# This forces Spark to redistribute the data into 4 chunks
df_repartitioned = df.repartition(4)

# 3. Save to a new parquet folder
output_path = "data/yellow_tripdata_2025_11_repartitioned"
df_repartitioned.write.mode("overwrite").parquet(output_path)

print(f"Done! Check the folder: {output_path}")

Done! Check the folder: data/yellow_tripdata_2025_11_repartitioned


## Question 3: Count records

How many taxi trips were there on the 15th of November?

Consider only trips that started on the 15th of November.

- 62,610
- 102,340
- **162,604**
- 225,768

In [12]:
from pyspark.sql import functions as F

# 1. Filter the DataFrame

trips_on_15th = df.filter(F.to_date(df.tpep_pickup_datetime) == "2025-11-15")
# 2. Execute the count
result = trips_on_15th.count()

print(f"Question 3 Answer: There were {result} trips on November 15, 2025.")

Question 3 Answer: There were 162604 trips on November 15, 2025.


## Question 4: Longest trip

What is the length of the longest trip in the dataset in hours?

- 22.7
- 58.2
- **90.6**
- 134.5

In [13]:
from pyspark.sql import functions as F

# 1. Calculate the duration in seconds and convert to hours
# We use unix_timestamp to get the epoch seconds for each column
df_with_duration = df.withColumn(
    "duration_hours", 
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 3600
)

# 2. Find the maximum value in that new column
max_duration = df_with_duration.select(F.max("duration_hours")).collect()[0][0]

print(f"The longest trip duration is: {max_duration:.1f} hours")

The longest trip duration is: 90.6 hours


## Question 5: User Interface

Spark's User Interface which shows the application's dashboard runs on which local port?

- 80
- 443
- **4040**
- 8080

## Question 6: Least frequent pickup location zone

Load the zone lookup data into a temp view in Spark:

```bash
wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
```

Using the zone lookup data and the Yellow November 2025 data, what is the name of the LEAST frequent pickup location Zone?

- **Governor's Island/Ellis Island/Liberty Island**
- Arden Heights
- Rikers Island
- Jamaica Bay

In [15]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv -P data/raw/

--2026-03-09 03:45:42--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 

52.85.39.153, 52.85.39.97, 52.85.39.65, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|52.85.39.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘data/raw/taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-03-09 03:45:43 (121 MB/s) - ‘data/raw/taxi_zone_lookup.csv’ saved [12331/12331]



In [16]:
# Load the Zone Lookup CSV
df_zones = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("data/raw/taxi_zone_lookup.csv")

# Create the temporary view
df_zones.createOrReplaceTempView("zones")

In [18]:
df.createOrReplaceTempView("yellow_data")

least_frequent_zone = spark.sql("""
    SELECT 
        z.Zone, 
        COUNT(*) as trip_count
    FROM 
        yellow_data y
    JOIN 
        zones z ON y.PULocationID = z.LocationID
    GROUP BY 
        z.Zone
    ORDER BY 
        trip_count ASC
    LIMIT 1
""")

least_frequent_zone.show(truncate=False)

+---------------------------------------------+----------+
|Zone                                         |trip_count|
+---------------------------------------------+----------+
|Governor's Island/Ellis Island/Liberty Island|1         |
+---------------------------------------------+----------+

